In [1]:
from pathlib import Path
import pandas as pd

try:
    PROJECT_ROOT = Path(__file__).parent.parent
except NameError:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

data_processed = PROJECT_ROOT / "data" / "processed"

df = pd.read_parquet(data_processed / "makassar_features.parquet")

print("DATA:", df.shape)

DATA: (5089864, 13)


In [2]:
# buat variasi antar grid
import numpy as np

np.random.seed(42)

# assign factor per grid (fixed per lokasi)
grid_factor = pd.DataFrame({
    "grid_id": df["grid_id"].unique()
})

grid_factor["spatial_factor"] = np.random.uniform(0.8, 1.2, len(grid_factor))

# merge ke dataset
df = df.merge(grid_factor, on="grid_id", how="left")

print("SPATIAL FACTOR ADDED")

SPATIAL FACTOR ADDED


In [3]:
df["rainfall"] = df["rainfall"] * df["spatial_factor"]

print("RAIN ADJUSTED")

RAIN ADJUSTED


In [4]:
df = df.sort_values(["grid_id", "date"])

lags = [1, 3, 7, 14, 30]

for lag in lags:
    df[f"rain_lag_{lag}"] = df.groupby("grid_id")["rainfall"].shift(lag)

df["rain_7d_sum"] = (
    df.groupby("grid_id")["rainfall"]
    .rolling(7)
    .sum()
    .reset_index(level=0, drop=True)
)

df["rain_30d_sum"] = (
    df.groupby("grid_id")["rainfall"]
    .rolling(30)
    .sum()
    .reset_index(level=0, drop=True)
)

In [5]:
df = df.dropna()

print("FINAL SHAPE:", df.shape)

FINAL SHAPE: (5004844, 14)


In [6]:
print("=== SAMPLE ===")
df.sample(10, random_state=42)

=== SAMPLE ===


,grid_id,date,rainfall,extreme_rain,flood_label,month,rain_lag_1,rain_lag_3,rain_lag_7,rain_lag_14,rain_lag_30,rain_7d_sum,rain_30d_sum,spatial_factor
2104275,19498,2021-04-04,106.955994,1,1,4,12.500440,20.740931,25.957696,40.360842,14.356068,230.305338,815.129147,1.192474
457302,16803,2021-02-22,14.119423,0,0,2,3.129542,15.363677,17.479061,31.003551,17.278694,89.859062,519.197431,0.867797
1077058,17873,2021-07-08,47.192889,1,1,7,19.355996,52.057141,4.473758,7.663721,6.519458,179.148846,521.966834,0.816347
2407576,19948,2020-08-24,11.870967,0,0,8,27.833473,50.598533,24.657207,42.249482,53.510498,152.207815,591.128441,0.903073
308372,16483,2021-07-10,4.060860,0,0,7,18.436745,11.223281,16.857838,37.941473,2.723849,130.203293,519.695158,0.806635
508020,16914,2022-04-28,4.443950,0,0,4,13.506112,38.846117,38.164011,37.966311,37.656428,99.059972,529.399437,0.950233
3271903,21002,2021-11-18,2.555614,0,0,11,7.149354,21.184287,41.245974,12.553884,1.428598,87.675104,449.169377,0.848555
1405902,18436,2021-12-31,8.034531,0,0,12,12.126906,5.623737,26.015245,10.456481,8.253089,71.066341,603.163857,0.900720
1452789,18538,2022-07-10,9.825682,0,0,7,30.087049,22.815373,33.489723,43.581099,14.644406,127.953875,662.446111,0.980336
809363,17503,2021-04-08,33.434605,0,0,4,5.521874,48.169168,30.763710,8.892140,25.959170,242.223172,642.427229,1.110859


In [7]:
out_path = data_processed / "makassar_features_spatial.parquet"

df.to_parquet(out_path, index=False)

print("✔ SAVED:", out_path)

✔ SAVED: d:\GITHUB\Project\flood-ml-research\data\processed\makassar_features_spatial.parquet
